# Publication-Quality Figures — IEEE Transactions on Quantum Engineering

This notebook generates publication-ready figures for the QAOA Portfolio Optimization paper, formatted to **IEEE Transactions on Quantum Engineering (TQE)** specifications.

**IEEE figure specs:**
- Column widths: single = 3.5 in, double = 7.16 in
- Font: Helvetica / Arial, 8–10 pt (min 6 pt)
- Resolution: color > 300 dpi, line art > 600 dpi
- Line width: ≥ 0.5 pt
- Format: PDF vector preferred

**Figures:**
1. Budget Violations vs Lambda + Barren Plateau Analysis
2. Convergence Plots (Evaluation Energy & Budget Violation)
3. Timing Comparison (GA vs BF) + Optimization Time

$$
\omega^{-(N-1)\cdot(N-1) }
$$

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from cycler import cycler
from scipy.optimize import minimize
import plotly.graph_objects as go

In [ ]:
# ============================================================
# Publication Style — IEEE Transactions on Quantum Engineering
# ============================================================
# Inspired by high-quality physics publication figures
#
# IEEE specs:
#   Column widths : single = 3.5 in (88.9 mm), double = 7.16 in (182 mm)
#   Max figure    : 7.16 x 8.8 in
#   Fonts         : Helvetica / Arial / Times New Roman, 8-10 pt (min 6 pt)
#   Resolution    : color/grayscale > 300 dpi, line art > 600 dpi
#   Line width    : >= 0.5 pt
#   Format        : PDF / EPS (vector preferred)

COLORS = {
    'blue':      '#4C72B0',   # Muted steel blue
    'orange':    '#DD8452',   # Warm terracotta
    'green':     '#55A868',   # Sage green
    'red':       '#C44E52',   # Muted crimson
    'purple':    '#8172B3',   # Soft purple
    'brown':     '#937860',   # Warm brown
    'pink':      '#DA8BC3',   # Dusty pink
    'gray':      '#8C8C8C',   # Neutral gray
    'gold':      '#CCB974',   # Muted gold
    'teal':      '#64B5CD',   # Light teal
}

COLOR_CYCLE = [
    COLORS['blue'], COLORS['orange'], COLORS['green'],
    COLORS['red'], COLORS['purple'], COLORS['brown'],
    COLORS['pink'], COLORS['gray'],
]

LAYER_COLORS  = {5: COLORS['blue'], 7: COLORS['red'], 9: COLORS['green']}
LAYER_MARKERS = {5: 'o', 7: 's', 9: '^'}
MIXER_MARKERS = {
    'X':            's',
    'Preserving12': 'o',
}

MIXER_COLORS = {
    'X':            COLORS['blue'],
    'Preserving12': COLORS['red'],
    'Preserving24': COLORS['green'],
}

mixer_label = {
    'X':              'X mixer',
    'Preserving12':   r'Subspace-confined mixer ($K\!=\!12$)',
    'Preserving24':   r'Subspace-confined mixer ($K\!=\!24$)',
}

# ---- IEEE column widths (inches) ----
FIG_SINGLE = 3.5
FIG_DOUBLE = 7.16

MARKERS = ['o', 's', '^', 'D', 'v', 'p', '*', 'h']
MS = 4
LW = 1.5   # thicker lines like reference figures

plt.rcParams.update({
    # --- Fonts ---
    'font.family':         'sans-serif',
    'font.sans-serif':     ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':           9,
    'mathtext.fontset':    'dejavusans',
    # --- Axes ---
    'axes.linewidth':      0.8,
    'axes.labelsize':      9,
    'axes.titlesize':      10,
    'axes.titleweight':    'bold',
    'axes.prop_cycle':     cycler('color', COLOR_CYCLE),
    'axes.spines.top':     True,
    'axes.spines.right':   True,
    'axes.grid':           False,      # No grid — clean like reference
    'axes.axisbelow':      True,
    # --- Ticks ---
    'xtick.labelsize':     8,
    'ytick.labelsize':     8,
    'xtick.major.width':   0.6,
    'ytick.major.width':   0.6,
    'xtick.major.size':    4.0,
    'ytick.major.size':    4.0,
    'xtick.minor.visible': False,
    'ytick.minor.visible': False,
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.top':           True,
    'ytick.right':         True,
    # --- Legend ---
    'legend.fontsize':     8,
    'legend.frameon':      True,
    'legend.framealpha':   0.85,
    'legend.edgecolor':    '#CCCCCC',
    'legend.fancybox':     True,
    'legend.borderpad':    0.4,
    'legend.handlelength': 1.5,
    'legend.handletextpad': 0.5,
    'legend.columnspacing': 1.0,
    # --- Lines ---
    'lines.linewidth':     LW,
    'lines.markersize':    MS,
    # --- Figure / export ---
    'figure.dpi':          150,
    'savefig.dpi':         600,
    'savefig.bbox':        'tight',
    'savefig.pad_inches':  0.03,
    'savefig.format':      'pdf',
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
})

print('IEEE TQE style loaded (physics publication reference).')
print(f'  Single-col: {FIG_SINGLE}"  |  Double-col: {FIG_DOUBLE}"')
print(f'  Font: {plt.rcParams["font.family"]}, {plt.rcParams["font.size"]} pt')
print(f'  Grid: OFF  |  Line width: {LW} pt')

## Data Loading

In [ ]:
# ============================================================
# Configuration
# ============================================================
dirrs_approx  = ['./experiments_approx_Q2_andrew']
dirrs_plateau = ['./experiments_plateau_X_Q2']

plot_layers     = [5, 7, 9]
plot_asset      = 5
loss_plot_lambda = 0.0005
loss_plot_q     = 1.5
loss_plot_mixers = ['Preserving12']

lamb_plateau = 0.0005
q_plateau    = 1.5
ddof         = 1

report_col  = ['Mixer', 'lambda', 'layer', 'Assets', 'Exp', 'Qubits',
               'Approximate_ratio', 'Budget_Violations', 'MaxProb_ratio',
               'init_1_time', 'init_2_time', 'optim_time', 'epochs',
               'observe_time', 'q', 'Risk', 'Return', 'Budget']
plateau_col = ['layer', 'Exp', 'Assets', 'Qubits', 'N',
               'Sum_1', 'Sum_2', 'Coeff', 'Budget', 'lambda', 'q']

# ---- Efficient Frontier config ----
ef_N_ASSETS     = 5
ef_exp_idx      = 2
ef_boost_X      = 5000
ef_boost_P      = 10500
ef_layers_X     = [5, 7, 9]
ef_layers_P     = [5, 7]
is_X_MaxProb = False
is_P_MaxProb = False
is_X_RandSol = False
is_P_RandSol = False
ef_K            = 12          # Preserving mixer bases
ef_n_feasible   = 400000
ef_n_infeasible = 7500

enable_rand_feasible = True
enable_rand_infeasible_X = True
enable_rand_infeasible_P = False

In [ ]:
# ============================================================
# Load approximation-ratio data
# ============================================================
data_approx = None
data_obj_loss = {}  # (mixer, lambda, q, Assets, layer) -> list of [arr]

for dirr in dirrs_approx:
    if not os.path.isdir(dirr):
        print(f'WARNING: {dirr} not found, skipping'); continue
    for dir_name in sorted(os.listdir(dirr)):
        sub = f'{dirr}/{dir_name}'
        if not os.path.isdir(sub):
            continue
        try:
            layer_val  = int(dir_name.split('_p')[1].split('_L')[0])
            lambda_val = float(dir_name.split('_L')[1].split('_q')[0])
            q_val      = float(dir_name.split('_q')[1].split('_')[0])
        except (IndexError, ValueError):
            continue
        # Pre-select CSVs: prefer report_*_AR2.csv over plain report_*.csv when both exist
        _report_csvs = [f for f in os.listdir(sub)
                        if f.startswith('report_') and f.endswith('.csv')
                        and not f.endswith('_MaxProb.csv') and not f.endswith('_RandSol.csv')]
        _ar2_bases = {f[:-len('_AR2.csv')] for f in _report_csvs if f.endswith('_AR2.csv')}
        _selected  = {f for f in _report_csvs
                      if f.endswith('_AR2.csv') or f[:-len('.csv')] not in _ar2_bases}
        for fn in os.listdir(sub):
            fpath = f'{sub}/{fn}'
            if fn in _selected:
                df = pd.read_csv(fpath)
                mixer_type = fn.split('report_')[1].split('_boost')[0]
                df['layer']  = layer_val
                df['Mixer']  = mixer_type
                df['lambda'] = lambda_val if mixer_type == 'X' else 1
                df['q']      = q_val
                if len(df) > 0:
                    data_approx = df if data_approx is None else pd.concat([data_approx, df], ignore_index=True)
            elif fn.startswith('expectation_') and fn.endswith('.npz'):
                arrs = np.load(fpath)
                mixer_type = fn.split('expectation_')[1].split('_boost')[0]
                for key in sorted(arrs.files):
                    if key.count('_') != 1:
                        continue
                    asset_val = int(key.split('_')[0].lstrip('A'))
                    key_base  = (mixer_type, lambda_val, q_val, asset_val, layer_val)
                    arr = arrs[key][:, 1:3].copy()  # eval, budget
                    arr[:, 1] = np.sqrt(arr[:, 1] / lambda_val)
                    data_obj_loss.setdefault(key_base, []).append([arr])

if data_approx is not None:
    data_approx = data_approx.sort_values(by=['Mixer','lambda','layer','Assets','Exp']).reset_index(drop=True)
    data_approx = data_approx[report_col]
    print(f'Loaded {len(data_approx)} approx rows, {len(data_obj_loss)} loss keys')
else:
    print('WARNING: no approx data loaded')

In [ ]:
# ============================================================
# Load barren-plateau data
# ============================================================
data_plateau = None

for dirr in dirrs_plateau:
    if not os.path.isdir(dirr):
        print(f'WARNING: {dirr} not found, skipping'); continue
    for dir_name in sorted(os.listdir(dirr)):
        sub = f'{dirr}/{dir_name}'
        if not os.path.isdir(sub):
            continue
        try:
            layer_val  = int(dir_name.split('_p')[1].split('_L')[0])
            lambda_val = float(dir_name.split('_L')[1].split('_q')[0])
            q_val      = float(dir_name.split('_q')[1])
        except (IndexError, ValueError):
            continue
        if q_val != q_plateau or lambda_val != lamb_plateau:
            continue
        for fn in os.listdir(sub):
            if not fn.startswith('report_') or not fn.endswith('.csv'):
                continue
            df = pd.read_csv(f'{sub}/{fn}')
            df['lambda'] = lambda_val
            df['layer']  = layer_val
            df['q']      = q_val
            if len(df) > 0:
                data_plateau = df if data_plateau is None else pd.concat([data_plateau, df], ignore_index=True)

if data_plateau is not None:
    data_plateau = data_plateau.sort_values(by=['Exp','Assets']).reset_index(drop=True)
    data_plateau = data_plateau[plateau_col]
    print(f'Loaded {len(data_plateau)} plateau rows')
else:
    print('WARNING: no plateau data loaded (experiments_plateau_X_Q2 missing?)')

## Efficient Frontier Data

In [ ]:
# ============================================================
# Load Efficient Frontier data (single experiment)
# ============================================================
name_idx = f"A{ef_N_ASSETS}_E{ef_exp_idx}"

# ---- 1. Load P, ret, cov, budget from first available X experiment ----
_dirr = f'{dirrs_approx[0]}/exp_p{ef_layers_X[0]}_L{loss_plot_lambda}_q{loss_plot_q}_torch'
_npz  = np.load(f'{_dirr}/expectation_X_boost_{ef_boost_X}.npz')
_csv  = pd.read_csv(f'{_dirr}/report_X_boost_{ef_boost_X}.csv')

ef_P      = _npz[f'{name_idx}_P']
ef_ret    = _npz[f'{name_idx}_ret']
ef_cov    = _npz[f'{name_idx}_cov']
ef_budget = _csv.loc[(_csv['Assets'] == ef_N_ASSETS) & (_csv['Exp'] == ef_exp_idx), 'Budget'].values[0]

ef_P_b   = ef_P / ef_budget
ef_ret_b = ef_ret * ef_P_b
ef_cov_b = np.diag(ef_P_b) @ ef_cov @ np.diag(ef_P_b)

# ---- 2. Extract quantum-optimized points ----
points_optim = []  # (risk, return, mode_str, layer, violation)
eps_min_X, eps_max_X = np.inf, -np.inf

for p in ef_layers_X:
    dirr = f'{dirrs_approx[0]}/exp_p{p}_L{loss_plot_lambda}_q{loss_plot_q}_torch'
    df = pd.read_csv(f'{dirr}/report_X_boost_{ef_boost_X}{"_MaxProb" if is_X_MaxProb else "_RandSol" if is_X_RandSol else ""}.csv')
    row = df[(df['Assets'] == ef_N_ASSETS) & (df['Exp'] == ef_exp_idx)]
    for i in range(len(row)):
        risk      = row['Risk'].values[i] / loss_plot_q
        ret_final = row['Return'].values[i]
        violation = np.sqrt(row['Budget_Violations'].values[i] / loss_plot_lambda)
        eps_min_X = min(eps_min_X, violation)
        eps_max_X = max(eps_max_X, violation)
        points_optim.append((risk, ret_final, 'X', p, violation))

eps_min_P, eps_max_P = np.inf, -np.inf
for p in ef_layers_P:
    dirr = f'{dirrs_approx[0]}/exp_p{p}_L1_q{loss_plot_q}_torch'
    rpt  = f'report_Preserving{ef_K}_boost_{ef_boost_P}{"_MaxProb" if is_P_MaxProb else "_RandSol" if is_P_RandSol else ""}.csv'
    df = pd.read_csv(f'{dirr}/{rpt}')
    row = df[(df['Assets'] == ef_N_ASSETS) & (df['Exp'] == ef_exp_idx)]
    for i in range(len(row)):
        risk      = row['Risk'].values[i] / loss_plot_q
        ret_final = row['Return'].values[i]
        violation = np.sqrt(row['Budget_Violations'].values[i] / 1.0)
        eps_min_P = min(eps_min_P, violation)
        eps_max_P = max(eps_max_P, violation)
        points_optim.append((risk, ret_final, f'Preserving{ef_K}', p, violation))

# ---- 3. Efficient frontier bounds ----
def _find_bound_var(cov_mat, ret_vec, is_min=True):
    """Find min/max variance portfolio under sum(w)=1, 0<=w<=1."""
    n = len(cov_mat)
    sign = 1.0 if is_min else -1.0
    res = minimize(lambda w: sign * (w @ cov_mat @ w),
                   x0=np.ones(n) / n, method='COBYLA',
                   bounds=[(0, 1)] * n,
                   constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                   options={'maxiter': 10000})
    w = res.x
    return w @ ret_vec, w @ cov_mat @ w

def _find_return_at_risk(ret_vec, cov_mat, target_var, maximize=True):
    """Maximize (or minimize) return subject to sum(w)=1 and w'Cw = target_var."""
    n = len(ret_vec)
    sign = -1.0 if maximize else 1.0
    res = minimize(lambda w: sign * (ret_vec @ w),
                   x0=np.ones(n) / n, method='COBYLA',
                   bounds=[(0, 1)] * n,
                   constraints=[
                       {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                       {'type': 'eq', 'fun': lambda w: w @ cov_mat @ w - target_var},
                   ],
                   options={'maxiter': 10000})
    if not res.success:
        return np.nan
    return ret_vec @ res.x  # actual return from optimal weights

_, min_var = _find_bound_var(ef_cov, ef_ret, is_min=True)
_, max_var = _find_bound_var(ef_cov, ef_ret, is_min=False)

risk_grid = np.linspace(min_var, max_var, 30)
ef_upper = [(v, _find_return_at_risk(ef_ret, ef_cov, v, maximize=True))  for v in risk_grid]
ef_lower = [(v, _find_return_at_risk(ef_ret, ef_cov, v, maximize=False)) for v in risk_grid]
ef_upper = np.array([(r, v) for r, v in ef_upper if not np.isnan(v)])
ef_lower = np.array([(r, v) for r, v in ef_lower if not np.isnan(v)])

print(f'Efficient frontier: {len(ef_upper)} upper, {len(ef_lower)} lower points')
print(f'  Upper return range: [{ef_upper[:,1].min():.6f}, {ef_upper[:,1].max():.6f}]')
print(f'  Lower return range: [{ef_lower[:,1].min():.6f}, {ef_lower[:,1].max():.6f}]')

# ---- 4. Random portfolio sampling (isolated RNG) ----
rng = np.random.default_rng(42)

ef_ret_feas = np.empty(ef_n_feasible)
ef_cov_feas = np.empty(ef_n_feasible)
if enable_rand_feasible:
    for i in range(ef_n_feasible):
        x = rng.dirichlet(np.ones(len(ef_P)))
        eps = rng.uniform(-0.1, 0.1)
        n = (x * ef_budget * (1 + eps)) / ef_P
        ef_ret_feas[i] = ef_ret_b.T @ n
        ef_cov_feas[i] = n.T @ ef_cov_b @ n

# Infeasible — SC mixer eps range
ef_ret_inf_P = np.empty(ef_n_infeasible)
ef_cov_inf_P = np.empty(ef_n_infeasible)
if enable_rand_infeasible_P:
    for i in range(ef_n_infeasible):
        x = rng.dirichlet(np.ones(len(ef_P)))
        eps = rng.uniform(-eps_max_P, eps_max_P)
        n = (x * ef_budget * (1 + eps)) / ef_P
        ef_ret_inf_P[i] = ef_ret_b.T @ n
        ef_cov_inf_P[i] = n.T @ ef_cov_b @ n

# Infeasible — X mixer eps range
ef_ret_inf_X = np.empty(ef_n_infeasible)
ef_cov_inf_X = np.empty(ef_n_infeasible)
if enable_rand_infeasible_X:
    for i in range(ef_n_infeasible):
        x = rng.dirichlet(np.ones(len(ef_P)))
        eps = rng.uniform(-eps_max_X, eps_max_X)
        n = (x * ef_budget * (1 + eps)) / ef_P
        ef_ret_inf_X[i] = ef_ret_b.T @ n
        ef_cov_inf_X[i] = n.T @ ef_cov_b @ n

print(f'Quantum points: {len(points_optim)} ({len(ef_layers_X)} X + {len(ef_layers_P)} SC)')
print(f'Random portfolios: {ef_n_feasible} feasible, {ef_n_infeasible} infeasible-SC, {ef_n_infeasible} infeasible-X')

---
## Figure 1: Budget Violations vs $\lambda$ &ensp;+&ensp; Barren Plateau Analysis

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

fig, axes = plt.subplots(2, 1, figsize=(FIG_DOUBLE*0.8, 5.0))

# ---- (a) Budget Violations vs Lambda ----
ax = axes[0]
if data_approx is not None:
    df_lam = data_approx[data_approx['Assets'] == plot_asset].copy()
    df_lam['BV'] = np.sqrt(df_lam['Budget_Violations'] / df_lam['lambda'])
    df_lam = df_lam.groupby(['lambda', 'layer', 'Mixer']).mean(numeric_only=True).reset_index()
    all_lambs = np.array(sorted(df_lam.loc[df_lam['Mixer'] == 'X', 'lambda'].unique()))
    all_mixers = df_lam['Mixer'].unique()
    all_layers = sorted(df_lam['layer'].unique())

    # Color gradient for layers: blue->red for X, green shades for SC
    cmap_x  = plt.cm.get_cmap('coolwarm')
    cmap_sc = plt.cm.get_cmap('Greens')
    norm_layer = Normalize(vmin=min(all_layers) - 1, vmax=max(all_layers) + 1)

    for mixer in all_mixers:
        for layer_val in all_layers:
            sub = df_lam[(df_lam['layer'] == layer_val) & (df_lam['Mixer'] == mixer)]
            if mixer == 'X':
                c = cmap_x(norm_layer(layer_val))
                ls = '-'
            else:
                c = cmap_sc(0.4 + 0.5 * norm_layer(layer_val))
                ls = '--'
            m = LAYER_MARKERS.get(layer_val, 'D')
            mixer_short = 'SP' if mixer == 'X' else 'SC'
            lbl = f"{mixer_short}, $L$={layer_val}"
            if mixer == 'X':
                ax.plot(sub['lambda'], sub['BV'],
                        marker=m, color=c, ls=ls, ms=MS, lw=LW, label=lbl)
            # else:
            #     bv = sub['BV'].values[0]
            #     ax.plot(all_lambs, [bv]*len(all_lambs),
            #             marker=m, color=c, ls=ls, ms=MS, lw=LW, label=lbl)

    ax.axhline(0.1, color='black', ls=':', lw=1.5, alpha=1.0,
               label=r'$\varepsilon = 0.1$')
    ax.set_xscale('log')
    ax.set_ylim(bottom=0)

ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'Budget violation $\tilde{\epsilon}$')
ax.set_title('(a)', loc='left')
ax.legend(fontsize=6.5, ncol=4, loc='center right',
          handlelength=1.2, columnspacing=0.8)

# ---- (b) Barren Plateau: Variance vs Qubits ----
# Use color gradient: light -> dark as P increases
ax = axes[1]
if data_plateau is not None:
    df_var = data_plateau[data_plateau['Exp'] == 0].copy()
    df_var['var'] = (df_var['Sum_2'] / (df_var['N'] - ddof)
                     - df_var['Sum_1']**2 / (df_var['N'] * (df_var['N'] - ddof)))
    df_var = df_var.groupby(['Qubits', 'layer']).mean(numeric_only=True).reset_index()
    all_layers_p = sorted(df_var['layer'].unique())

    # Gradient colormap: plasma (yellow->purple as P increases)
    cmap_bp = plt.cm.get_cmap('plasma')
    norm_p = Normalize(vmin=min(all_layers_p) - 1, vmax=max(all_layers_p) + 1)

    for i, lv in enumerate(all_layers_p):
        sub = df_var[df_var['layer'] == lv]
        c = cmap_bp(norm_p(lv))
        m = MARKERS[i % len(MARKERS)]
        ax.plot(sub['Qubits'], sub['var'],
                marker=m, color=c, ms=MS, lw=LW, label=f'SP, $L$={lv}')

    ax.set_yscale('log')
    ax.legend(fontsize=6.5, ncol=2,
              handlelength=1.2, columnspacing=0.8)
else:
    ax.text(0.5, 0.5, 'Plateau data\nnot available',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=9, color='grey')

ax.set_xlabel('Number of qubits')
ax.set_ylabel(r'$\underset{\boldsymbol{\beta}, \boldsymbol{\gamma}}{\mathrm{Var}}[\langle Z_2Z_3 \rangle]$')
ax.set_title('(b)', loc='left')

fig.tight_layout(w_pad=3.0)
# fig.savefig('Fig_1.pdf')
fig.savefig('Fig_1_UpFont_NoSC.png', dpi=600)
plt.show()
# print('Saved Fig_1.pdf / Fig_1.png')
# print(data_plateau["lambda"])

---
## Figure 2: Convergence Plots (Evaluation Energy & Budget Violation)

In [ ]:
n_rows = len(loss_plot_mixers)
fig, axes = plt.subplots(n_rows, 2, figsize=(FIG_DOUBLE, 2.8 * n_rows), squeeze=False)

obj_loss = {k: [x.copy() for x in v] for k, v in data_obj_loss.items()}  # deep copy

for row, mixer in enumerate(loss_plot_mixers):
    lam_use = loss_plot_lambda if mixer == 'X' else 1.0

    for ci, lv in enumerate(plot_layers):
        key = (mixer, lam_use, loss_plot_q, plot_asset, lv)
        if key not in obj_loss:
            print(f'  No data for {key}'); continue

        # pad to equal length
        max_len = max(a.shape[0] for lst in obj_loss[key] for a in lst)
        for lst in obj_loss[key]:
            for j in range(len(lst)):
                a = lst[j]
                if a.shape[0] < max_len:
                    pad = np.tile(a[-1], (max_len - a.shape[0], 1))
                    lst[j] = np.vstack([a, pad])

        arr = np.array(obj_loss[key]).squeeze(1)  # (n_exp, T, 2)

        # mean +/- 1 STD
        med = np.mean(arr, axis=0)
        std = np.std(arr, axis=0)
        lo  = med - std
        hi  = med + std
        t   = np.arange(med.shape[0])

        c = LAYER_COLORS.get(lv, COLOR_CYCLE[ci])
        lbl = f'$P$ = {lv}'

        for col in range(2):
            ax = axes[row, col]
            # Shaded band (like reference figure)
            # ax.fill_between(t, lo[:, col], hi[:, col],
            #                 color=c, alpha=0.20)
            # Mean line on top
            ax.plot(t, med[:, col], color=c, lw=LW, label=lbl)

    # axis labels
    for col, ylabel in enumerate(['Evaluation energy', 'Budget violation']):
        ax = axes[row, col]
        ax.set_xlabel('Training iteration')
        ax.set_ylabel(ylabel)
        panel = chr(ord('a') + 2 * row + col)
        ax.set_title(f'({panel})', loc='left')
        if col == 1:  # budget violation panel
            ax.axhline(0.1, color='gray', ls=':', lw=1.0,
                       alpha=0.5, label=r'$\varepsilon = 0.1$')
        ax.legend(fontsize=7)
        # ax.set_ylim()

# ---- Equalize y-axis ranges for evaluation energy (a)=(c) only ----
# Budget violation panels (b) and (d) keep independent scales
col = 0  # evaluation energy column
ylims = [axes[row, col].get_ylim() for row in range(n_rows)]
shared_lo = min(lo for lo, hi in ylims)
shared_hi = max(hi for lo, hi in ylims)
for row in range(n_rows):
    axes[row, col].set_ylim(shared_lo, shared_hi)

fig.tight_layout(w_pad=2.5, h_pad=3.0)
fig.savefig('Fig_2.pdf')
fig.savefig('Fig_2.png', dpi=600)
plt.show()
print('Saved Fig_2.pdf / Fig_2.png')

In [ ]:
n_rows = len(loss_plot_mixers)
fig, axes = plt.subplots(2, 1, figsize=(FIG_DOUBLE*0.8, 5.0), squeeze=False)

obj_loss = {k: [x.copy() for x in v] for k, v in data_obj_loss.items()}  # deep copy

for row, mixer in enumerate(loss_plot_mixers):
    lam_use = loss_plot_lambda if mixer == 'X' else 1.0

    for ci, lv in enumerate(plot_layers):
        key = (mixer, lam_use, loss_plot_q, plot_asset, lv)
        if key not in obj_loss:
            print(f'  No data for {key}'); continue

        # pad to equal length
        max_len = max(a.shape[0] for lst in obj_loss[key] for a in lst)
        for lst in obj_loss[key]:
            for j in range(len(lst)):
                a = lst[j]
                if a.shape[0] < max_len:
                    pad = np.tile(a[-1], (max_len - a.shape[0], 1))
                    lst[j] = np.vstack([a, pad])

        arr = np.array(obj_loss[key]).squeeze(1)  # (n_exp, T, 2)

        # mean +/- 1 STD
        med = np.mean(arr, axis=0)
        std = np.std(arr, axis=0)
        lo  = med - std
        hi  = med + std
        t   = np.arange(med.shape[0])

        c = LAYER_COLORS.get(lv, COLOR_CYCLE[ci])
        mixer_short = 'SP' if mixer == 'X' else 'SC'
        lbl = f"{mixer_short}, $L$={lv}"

        for col in range(2):
            ax = axes[1-col, row]
            # Shaded band (like reference figure)
            # ax.fill_between(t, lo[:, col], hi[:, col],
            #                 color=c, alpha=0.20)
            # Mean line on top
            ax.plot(t, med[:, col], color=c, lw=LW, label=lbl)

    # axis labels
    for col, ylabel in enumerate([r'$\langle H_{\text{obj}} \rangle$', r'Budget violation $\tilde{\epsilon}$']):
        ax = axes[1-col, row]
        ax.set_xlabel('Training iteration')
        ax.set_ylabel(ylabel)
        panel = chr(ord('a') + 2 * row + 1-col)
        ax.set_title(f'({panel})', loc='left')
        if col == 1:  # budget violation panel
            ax.axhline(0.1, color='black', ls=':', lw=1.5,
                       alpha=1.0, label=r'$\varepsilon = 0.1$')
            ax.set_ylim(bottom=-0.0, top=0.3)
        ax.legend(fontsize=7)

# ---- Equalize y-axis ranges for evaluation energy (a)=(c) only ----
# Budget violation panels (b) and (d) keep independent scales
col = 0  # evaluation energy column
ylims = [axes[col, row].get_ylim() for row in range(n_rows)]
shared_lo = min(lo for lo, hi in ylims)
shared_hi = max(hi for lo, hi in ylims)
for row in range(n_rows):
    axes[col, row].set_ylim(shared_lo, shared_hi)

fig.tight_layout(w_pad=2.5, h_pad=3.0)
# fig.savefig('Fig_2.pdf')
fig.savefig('Fig_2_UpFont_noBound.png', dpi=600)
plt.show()
print('Saved Fig_2.pdf / Fig_2.png')

---
## Figure 3: GA vs Brute Force — Time & Budget Violation

In [ ]:
speed_file = './speed.csv'
has_speed  = os.path.isfile(speed_file)

if has_speed:
    df_sp = pd.read_csv(speed_file).groupby('Assets').mean(numeric_only=True).reset_index()

    fig, axes = plt.subplots(2, 1, figsize=(FIG_DOUBLE*0.8, 5.0))

    # ---- (a) GA vs BF time ----
    ax = axes[0]
    modes_sp = [('GA_time_ms', 'Genetic algorithm', COLORS['blue'],   'o'),
                ('BF_time_ms', 'Brute force',       COLORS['orange'], 's')]
    for col, lbl, c, m in modes_sp:
        ax.semilogy(df_sp['Assets'], df_sp[col],
                    marker=m, color=c, ms=MS, lw=LW, label=lbl)
    ax.set_xlabel('Number of assets ($N$)')
    ax.set_ylabel('Time (ms)')
    ax.set_title('(a)', loc='left')
    ax.legend(fontsize=7)

    # ---- (b) Budget violation: GA vs BF ----
    ax = axes[1]
    eps_modes = [
        ('mean12_eps_GA', r'GA ($K$=12)',  COLORS['blue'],   'o'),
        ('mean12_eps_BF', r'BF ($K$=12)',  COLORS['orange'], 's'),
        ('mean24_eps_GA', r'GA ($K$=24)',  COLORS['green'],  '^'),
        ('mean24_eps_BF', r'BF ($K$=24)',  COLORS['red'],    'D'),
    ]
    for col, lbl, c, m in eps_modes:
        if col in df_sp.columns:
            ax.semilogy(df_sp['Assets'], df_sp[col],
                        marker=m, color=c, ms=MS, lw=LW, label=lbl)
    ax.axhline(0.1, color='black', ls=':', lw=1.5,
               alpha=1.0, label=r'$\varepsilon = 0.1$')
    ax.set_xlabel('Number of assets ($N$)')
    ax.set_ylabel(r'Budget violation $\Delta$')
    ax.set_title('(b)', loc='left')
    ax.legend(fontsize=6)

    fig.tight_layout(w_pad=3.0)
    # fig.savefig('Fig_3.pdf')
    fig.savefig('Fig_3_UpFont.png', dpi=600)
    plt.show()
    print('Saved Fig_3.pdf / Fig_3.png')
else:
    print('speed.csv not found — skipping Figure 3.')

In [ ]:
# ============================================================
# Figure 4: Optimization Time vs Assets
# ============================================================
L_time      = 0.0005
mixers_time = ['X', 'Preserving12']
layers_time = [5, 7, 9]

if data_approx is not None:
    fig, ax = plt.subplots(figsize=(FIG_SINGLE * 1.4, FIG_SINGLE * 1.1))
    df_t = data_approx[data_approx['lambda'] == L_time].copy()
    df_t = df_t.groupby(['Assets', 'Mixer', 'layer']).mean(numeric_only=True).reset_index()

    cmap_warm = plt.cm.get_cmap('Oranges')
    cmap_cool = plt.cm.get_cmap('Blues')
    ci = 0
    for mixer in mixers_time:
        cmap = cmap_warm if mixer == 'X' else cmap_cool
        for li, lv in enumerate(layers_time):
            sub = df_t[(df_t['Mixer'] == mixer) & (df_t['layer'] == lv)]
            if len(sub) == 0:
                ci += 1; continue
            frac = 0.4 + 0.5 * li / max(len(layers_time) - 1, 1)
            c = cmap(frac)
            m = MARKERS[ci % len(MARKERS)]
            mixer_short = 'SP' if mixer == 'X' else 'SC'
            lbl = f"{mixer_short}, $P$={lv}"
            ax.semilogy(sub['Assets'], sub['optim_time'],
                        marker=m, color=c, ms=MS, lw=LW, label=lbl)
            ci += 1
    ax.set_xlabel('Number of assets ($N$)')
    ax.set_ylabel('Optimization time (s)')
    ax.legend(fontsize=6.5, loc='upper left')
    fig.tight_layout()
    fig.savefig('Fig_4.pdf')
    fig.savefig('Fig_4.png', dpi=600)
    plt.show()
    print('Saved Fig_4.pdf / Fig_4.png')
else:
    print('No approx data — skipping Figure 4.')

---
## Figure 5: Efficient Frontier — Feasible & Infeasible Portfolios

In [ ]:
# ============================================================
# Figure 5: Efficient Frontier
# ============================================================
fig, ax = plt.subplots(figsize=(FIG_DOUBLE*0.8, FIG_DOUBLE * 0.6))

# ---- Infeasible scatter: X mixer eps range ----
# ax.scatter(ef_cov_inf_X, ef_ret_inf_X,
#            s=6, alpha=0.25, color=COLORS['gold'], marker='^',
#            rasterized=True, zorder=1)
# ax.scatter([], [], s=20, alpha=0.8, color=COLORS['gold'], marker='^',
#            label=f'Infeasible (SP, $\\varepsilon \\in [{eps_min_X:.3f},\\, {eps_max_X:.3f}]$)')

# ---- Infeasible scatter: SC mixer eps range ----
# ax.scatter(ef_cov_inf_P, ef_ret_inf_P,
#            s=6, alpha=0.25, color=COLORS['purple'], marker='^',
#            rasterized=True, zorder=1)
# ax.scatter([], [], s=20, alpha=0.8, color=COLORS['purple'], marker='^',
#            label=f'Infeasible (SC, $\\varepsilon \\in [{eps_min_P:.3f},\\, {eps_max_P:.3f}]$)')

# ---- Background scatter: feasible portfolios ----
ax.scatter(ef_cov_feas, ef_ret_feas,
           s=6, alpha=0.25, color=COLORS['teal'], marker='^',
           rasterized=True, zorder=1)
ax.scatter([], [], s=20, alpha=0.8, color=COLORS['teal'], marker='^',
           label=r'Random admissible portfolios $\varepsilon=0.1$')

# # ---- Efficient frontier bounds ----
# if len(ef_upper) > 0:
#     ax.plot(ef_upper[:, 0], ef_upper[:, 1],
#             color=COLORS['red'], ls='--', lw=LW, zorder=3,
#             label='Efficient frontier (upper)')
# if len(ef_lower) > 0:
#     ax.plot(ef_lower[:, 0], ef_lower[:, 1],
#             color=COLORS['brown'], ls='--', lw=LW, zorder=3,
#             label='Efficient frontier (lower)')

# ---- Quantum-optimized points ----
# Offset annotations to avoid overlap
annot_offsets = {
    ('X', 5):             (6, 0),
    ('X', 7):             (6, 0),
    ('X', 9):             (6, 0),
    ('Preserving12', 5):  (-60, 10),
    ('Preserving12', 7):  (-70, 5),
    ('Preserving12', 9):  (-60, 20),
}
for risk, ret_val, mode, layer, violation in points_optim:
    if mode == 'X':
        c = LAYER_COLORS[layer]
        lbl = f'SP, $L$={layer}'
    else:
        c = LAYER_COLORS[layer]
        lbl = f'SC ($K$={ef_K}), $L$={layer}'
    # m = LAYER_MARKERS.get(layer, 'D')
    m = MIXER_MARKERS.get(mode, 'D')
    ax.scatter(risk, ret_val, color=c, s=40, marker=m, zorder=5,
               edgecolors='black', linewidths=0.5, label=lbl)
    ofs = annot_offsets.get((mode, layer), (5, -8))
    ax.annotate(f'$\\tilde{{\\epsilon}}$={violation:.3f}',
                xy=(risk, ret_val),
                xytext=ofs, textcoords='offset points',
                fontsize=7, color=c,
                arrowprops=dict(arrowstyle='-', color=c, lw=0.5) if abs(ofs[0]) > 20 else None)

# ---- Axes ----
ax.set_xlabel('Portfolio risk (variance)')
ax.set_ylabel('Portfolio return')

ax.legend(fontsize=7, ncol=1,
          handlelength=1.2, columnspacing=0.8,
          framealpha=0.85)

fig.tight_layout()
# fig.savefig('Fig_5.pdf')
fig.savefig('Fig_4_UpFont.png', dpi=600)
plt.show()
print('Saved Fig_4.pdf / Fig_4.png')

---
## Figure 6: 3D Efficient Frontier (Interactive)

In [ ]:
# ============================================================
# Figure 6: 3D Efficient Frontier (Interactive — Plotly)
# ============================================================
# Stacks three layers along the z-axis:
#   z = 0  Feasible random portfolios
#   z = 1  Infeasible (SC mixer epsilon range)
#   z = 2  Infeasible (SP mixer epsilon range)
# Quantum-optimized points are plotted at their respective layer heights.

fig6 = go.Figure()

layer_height = 1          # spacing between z-layers
now_z = 0
z_labels = {}

# ---- Layer 0: Feasible portfolios ----
z_labels[now_z] = 'Feasible'
fig6.add_trace(go.Scatter3d(
    x=ef_cov_feas, y=ef_ret_feas,
    z=np.full(len(ef_ret_feas), now_z),
    mode='markers',
    marker=dict(size=3, color=COLORS['teal'], opacity=0.10),
    name='Feasible random portfolios',
))
now_z += layer_height

# ---- Layer 1: Infeasible — SC mixer ----
z_labels[now_z] = 'SC mixer'
fig6.add_trace(go.Scatter3d(
    x=ef_cov_inf_P, y=ef_ret_inf_P,
    z=np.full(len(ef_ret_inf_P), now_z),
    mode='markers',
    marker=dict(size=3, color=COLORS['purple'], opacity=0.10),
    name=f'Infeasible (SC, \u03b5\u2208[{eps_min_P:.3f}, {eps_max_P:.3f}])',
))
now_z += layer_height

# ---- Layer 2: Infeasible — SP mixer ----
z_labels[now_z] = 'SP mixer'
fig6.add_trace(go.Scatter3d(
    x=ef_cov_inf_X, y=ef_ret_inf_X,
    z=np.full(len(ef_ret_inf_X), now_z),
    mode='markers',
    marker=dict(size=3, color=COLORS['gold'], opacity=0.10),
    name=f'Infeasible (SP, \u03b5\u2208[{eps_min_X:.3f}, {eps_max_X:.3f}])',
))

# ---- Quantum-optimized points ----
for risk, ret_val, mode, layer, violation in points_optim:
    if mode == 'X':
        z_val = 2 * layer_height   # SP mixer layer
        c = LAYER_COLORS[layer]
        lbl = f'SP, P={layer}'
    else:
        z_val = 1 * layer_height   # SC mixer layer
        c = COLORS['green']
        lbl = f'SC (K={ef_K}), P={layer}'

    fig6.add_trace(go.Scatter3d(
        x=[risk], y=[ret_val], z=[z_val],
        mode='markers+text',
        marker=dict(size=6, color=c, symbol='x',
                    line=dict(width=1, color='black')),
        text=[f'\u03b5={violation:.3f}'],
        textposition='top center',
        textfont=dict(size=9, color=c),
        name=lbl,
    ))

# ---- Layout ----
fig6.update_layout(
    title='3D Efficient Frontier — Feasible & Infeasible Portfolios',
    scene_aspectmode='manual',
    scene_aspectratio=dict(x=5, y=5, z=2),
    scene=dict(
        xaxis_title='Portfolio Risk (Variance)',
        yaxis_title='Portfolio Return',
        zaxis_title='',
        zaxis=dict(
            tickvals=list(z_labels.keys()),
            ticktext=list(z_labels.values()),
        ),
        camera=dict(
            center=dict(x=0, y=0, z=0),
            eye=dict(x=0, y=-3, z=5),
        ),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig6.show()
print('Figure 6: 3D Efficient Frontier displayed (interactive).')

---


## Approximate Ratio vs Assets — Lambda sweep


In [ ]:
# ============================================================
# Approximate Ratio vs Assets — Lambda sweep
# ============================================================
# Reproduces the AR-vs-Assets plots from plot.ipynb (cells 43-45)
# in the IEEE TQE publication style. One figure per lambda;
# SC (Preserving12) curves are identical across lambdas (one
# experiment set), while SP (X-mixer) curves are lambda-specific.
ar_lambdas = [0.0005, 5]
ar_mixers  = ['X', 'Preserving12']
ar_layers  = [5, 7, 9]
ar_q       = 1.5

if data_approx is not None:
    for L_ar in ar_lambdas:
        fig, ax = plt.subplots(figsize=(FIG_SINGLE * 1.4, FIG_SINGLE * 1.1))

        df_X = data_approx[(data_approx['Mixer'] == 'X')
                           & (data_approx['lambda'] == L_ar)
                           & (data_approx['q'] == ar_q)]
        df_P = data_approx[(data_approx['Mixer'] == 'Preserving12')
                           & (data_approx['lambda'] == 1)
                           & (data_approx['q'] == ar_q)]
        df_ar = pd.concat([df_X, df_P], ignore_index=True)
        df_ar = (df_ar.groupby(['Assets', 'Mixer', 'layer'])
                       .mean(numeric_only=True)
                       .reset_index())

        cmap_warm = plt.cm.get_cmap('Oranges')
        cmap_cool = plt.cm.get_cmap('Blues')
        ci = 0
        for mixer in ar_mixers:
            cmap = cmap_warm if mixer == 'X' else cmap_cool
            for li, lv in enumerate(ar_layers):
                sub = df_ar[(df_ar['Mixer'] == mixer) & (df_ar['layer'] == lv)]
                if len(sub) == 0:
                    ci += 1
                    continue
                frac = 0.4 + 0.5 * li / max(len(ar_layers) - 1, 1)
                c = cmap(frac)
                m = MARKERS[ci % len(MARKERS)]
                mixer_short = 'SP' if mixer == 'X' else 'SC'
                lbl = f"{mixer_short}, $P$={lv}"
                ax.plot(sub['Assets'], sub['Approximate_ratio'],
                        marker=m, color=c, ms=MS, lw=LW, label=lbl)
                ci += 1

        ax.set_xlabel('Number of assets ($N$)')
        ax.set_ylabel('Approximate ratio')
        ax.set_ylim(0, 1)
        ax.set_title(rf'$Q={ar_q},\ \lambda={L_ar}$')
        ax.legend(fontsize=6.5, loc='best', ncol=2)
        fig.tight_layout()

        l_tag = str(L_ar).replace('.', 'p')
        fig.savefig(f'Fig_AR_L{l_tag}.pdf')
        fig.savefig(f'Fig_AR_L{l_tag}.png', dpi=600)
        plt.show()
        print(f'Saved Fig_AR_L{l_tag}.pdf / .png  (lambda={L_ar})')
else:
    print('data_approx not loaded — run the approximation-ratio loader cell first.')
